# GSE279942 재분석: nCRT 전 직장암 생검에서 반응군별 면역글로불린(IG) 신호

**목적**: nCRT 전 직장암 생검에서 형질세포·면역글로불린(IG) 신호가 병리 반응(TRG)과 연관되는지 공개 데이터로 재분석

**데이터**: GSE279942 (Corrò et al., *iScience* 2026, doi 10.1016/j.isci.2025.114455)
- nCRT 전 직장암 생검 51명, GeoMx WTA (FFPE TMA)
- 병리의사가 고른 ROI를 PanCK 염색으로 **종양 구획(PanCK+)** / **기질 구획(PanCK−)** 으로 나눠 측정 → 세포 단위가 아닌 구획 단위
- 반응: 좋은 반응(Good, TRG 1–2) / 중간(Mid, TRG 3) / 무반응(Non, TRG 3-4 이상)

**분석 원칙**
- 통계 단위는 환자 (같은 환자의 여러 코어는 평균)
- 주요 지표는 결과를 보기 전에 정함: **기질 IgG ↔ TRG**
- 원 논문이 올린 xlsx 행렬은 유전자 10,003개만 남아 있어서(검출률 낮은 유전자 제거), GEO 원본 카운트(DCC) + 프로브 표(PKC)로 전체 행렬을 다시 만들어 사용

**실행 순서**: 위에서부터 차례로 실행. 결과는 `BASE/final/`에 저장됨 (1번 셀에서 `BASE` 경로를 원하는 곳으로 바꿔서 사용)

## 1. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, gzip, json, glob, tarfile, subprocess, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from scipy import stats
import statsmodels.formula.api as smf

SEED = 0                     # 그림의 점 흩뿌리기(jitter)에만 쓰임. 통계 값은 무작위성 없음
np.random.seed(SEED)

BASE = '/content/drive/MyDrive/GSE279942_reanalysis'   # 원하는 드라이브 경로로 수정
RAW = f'{BASE}/geo_raw'      # GEO 원본 파일 보관
OUT = f'{BASE}/final'        # 최종 결과 저장
os.makedirs(RAW, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
GEO = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE279nnn/GSE279942'

# 한글 폰트 (코랩 기본 폰트엔 한글이 없음)
if not glob.glob('/usr/share/fonts/truetype/nanum/NanumGothic*.ttf'):
    subprocess.run('apt-get -qq install -y fonts-nanum', shell=True)
for fp in glob.glob('/usr/share/fonts/truetype/nanum/NanumGothic*.ttf'):
    fm.fontManager.addfont(fp)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

## 2. GEO 원본 파일 받기
- `series_matrix`: 샘플(segment)별 메타데이터 (환자, 구획, 반응, TRG, 슬라이드, 핵 수)
- `RAW.tar`: segment별 원본 카운트 파일(DCC) 97개
- `pkc`: 프로브 ID → 유전자 이름 표

한 번 받으면 드라이브에 저장되어 다음부터는 건너뜀. 받은 파일의 SHA-256 지문을 기록해서, 나중에 GEO 파일이 바뀌었는지 비교할 수 있게 함

In [ ]:
URLS = {'series_matrix': f'{GEO}/matrix/GSE279942_series_matrix.txt.gz',
        'raw_tar':       f'{GEO}/suppl/GSE279942_RAW.tar',
        'pkc':           f'{GEO}/suppl/GSE279942_Hs_R_NGS_WTA_v1.0.pkc.gz'}
paths = {}
for k, url in URLS.items():
    paths[k] = f'{RAW}/{url.split("/")[-1]}'
    if not os.path.exists(paths[k]):
        print('다운로드:', url)
        urllib.request.urlretrieve(url, paths[k])
# 파일 지문(SHA-256): GEO 파일이 나중에 바뀌었는지 확인하는 용도
import hashlib
def sha256(fp):
    h = hashlib.sha256()
    with open(fp, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

file_hash = {k: sha256(v) for k, v in paths.items()}
for k, v in file_hash.items():
    print(f'{k:<14} {v}')

## 3. 메타데이터 정리
- segment 97개 = 종양 구획 50 + 기질 구획 47
- `CRCHUG_096_1`, `CRCHUG_096_2`는 같은 환자의 코어 2개라서 `CRCHUG_096`으로 합침 → 환자 51명
- TRG를 숫자로 바꿈 (`TRG 2-3` → 2.5 같은 식). 단계적 경향(좋은 반응 → 중간 → 무반응)을 보기 위함

In [ ]:
cols = {}
with gzip.open(paths['series_matrix'], 'rt', encoding='utf-8', errors='replace') as f:
    for line in f:
        p = [x.strip('"') for x in line.rstrip('\n').split('\t')]
        if p[0] == '!Sample_title':
            cols['title'] = p[1:]
        elif p[0] == '!Sample_geo_accession':
            cols['GSM'] = p[1:]
        elif p[0] == '!Sample_characteristics_ch1':
            cols[p[1].split(': ', 1)[0]] = [v.split(': ', 1)[-1] for v in p[1:]]
raw_meta = pd.DataFrame(cols)

TRG = {'TRG 1': 1, 'TRG 2': 2, 'TRG 2-3': 2.5, 'TRG 3': 3, 'TRG 3-4': 3.5,
       'TRG 4': 4, 'TRG 4-5': 4.5, 'TRG 5': 5}
meta = pd.DataFrame({
    'GSM':     raw_meta['GSM'],
    'patient': raw_meta['code'].str.extract(r'^(CRCHUG_\d+)')[0],
    'seg':     raw_meta['segmentlabel'],                          # tumor / stroma
    'group':   raw_meta['response'].str.split().str[0],           # Good / Mid / Non
    'trg':     raw_meta['mandard_equivalent'].map(TRG),           # NA는 결측
    'slide':   raw_meta['slidename'],
    'nuclei':  pd.to_numeric(raw_meta['aoinucleicount']),
})
print('segment 수 (반응군 × 구획)')
print(pd.crosstab(meta.group, meta.seg))
print('\n환자 수:', meta.groupby('group').patient.nunique().to_dict())

## 4. 원본 카운트로 전체 유전자 행렬 만들기 + Q3 정규화
- DCC 파일의 프로브별 카운트를 PKC 표로 유전자 이름에 연결
- 음성 대조 프로브(`NegProbe`)는 제외
- **Q3 정규화**: segment마다 상위 25% 지점(Q3) 값으로 나눠 segment 크기·시퀀싱 깊이 차이를 맞춤 (GeoMx 표준, 원 논문과 같은 방식)

In [ ]:
with gzip.open(paths['pkc'], 'rt') as f:
    pkc = json.load(f)
rts2gene = {p['RTS_ID']: t['DisplayName'] for t in pkc['Targets'] for p in t['Probes']}

dcc_dir = '/content/gse279942_dcc'
if not glob.glob(f'{dcc_dir}/*.dcc.gz'):
    os.makedirs(dcc_dir, exist_ok=True)
    with tarfile.open(paths['raw_tar']) as t:
        try:
            t.extractall(dcc_dir, filter='data')
        except TypeError:          # 옛 파이썬 버전
            t.extractall(dcc_dir)

counts = {}
for fp in sorted(glob.glob(f'{dcc_dir}/*.dcc.gz')):
    gsm = os.path.basename(fp).split('_')[0]
    with gzip.open(fp, 'rt') as f:
        block = re.search(r'<Code_Summary>(.*?)</Code_Summary>', f.read(), re.S).group(1)
    counts[gsm] = {k: float(v) for k, v in (ln.split(',') for ln in block.split() if ',' in ln)}

probe = pd.DataFrame(counts).fillna(0)
probe = probe[probe.index.isin(rts2gene)]
gene = probe.groupby(probe.index.map(rts2gene)).sum()
gene = gene.drop(index=[g for g in gene.index if 'NegProbe' in g])

q3 = gene.quantile(0.75)
norm = gene / q3 * np.exp(np.log(q3).mean())
print('정규화 행렬 (유전자 × segment):', norm.shape)

## 5. 세포 신호 점수 → 환자 단위 표
점수 = 해당 유전자들의 log2(Q3 정규화 값 + 1) 평균

| 점수 | 유전자 | 비고 |
|---|---|---|
| IgG | IGHG1–4 | 주요 지표 (기질 구획) |
| IgA | IGHA1 | IgG만의 변화인지 확인용 |
| IGKC | IGKC | 경쇄. 검출률 99%로 가장 안정적인 형질세포 신호 |
| CD8 | CD8A, CD8B | 탐색적. 대부분 segment에서 배경 수준이라 해석 주의 |

IGHGP, IGHA2, IGLC는 이 플랫폼(PKC)에 아예 없음

In [ ]:
GENES = {'IgG':  ['IGHG1', 'IGHG2', 'IGHG3', 'IGHG4'],
         'IgA':  ['IGHA1'],
         'IGKC': ['IGKC'],
         'CD8':  ['CD8A', 'CD8B']}
for k, g in GENES.items():
    meta[k] = meta['GSM'].map(np.log2(norm.loc[g] + 1).mean())

# 환자 × 구획으로 평균 (096처럼 코어가 여러 개면 평균), 핵 수는 합, 슬라이드는 첫 값
seg_vals = meta.groupby(['patient', 'seg']).agg({**{k: 'mean' for k in GENES}, 'nuclei': 'sum', 'slide': 'first'})
P = seg_vals.unstack('seg')
P.columns = [f'{a}_{b}' for a, b in P.columns]          # 예: IgG_stroma, nuclei_tumor
P = meta.groupby('patient')[['group', 'trg']].first().join(P)
P['CD8_tumor_minus_stroma'] = P['CD8_tumor'] - P['CD8_stroma']   # 낮을수록 기질 쪽에 몰림
P.to_csv(f'{OUT}/patient_scores.csv')
P.head()

## 6. 통계
- **TRG 경향**: 점수와 TRG의 스피어만 상관 (중간 반응군까지 51명 전체 사용)
- **보정 p**: 선형회귀 `점수 ~ TRG + 슬라이드 + log(핵 수)`. 4번의 슬라이드 런과, 무반응군 종양 segment가 작은(핵 수가 적은) 차이를 보정
- **두 군 비교**: Mann-Whitney U (다중비교 보정 안 함)
- **종양 vs 기질**: 같은 환자끼리 짝지은 Wilcoxon 부호순위 검정

In [ ]:
ORDER = ['Good', 'Mid', 'Non']
PANELS = [('IgG_stroma', '기질 IgG'), ('IGKC_stroma', '기질 IGKC'), ('IgA_stroma', '기질 IgA'),
          ('IGKC_tumor', '종양 구획 IGKC'), ('CD8_tumor_minus_stroma', 'CD8 종양 - 기질 (탐색적)')]


def adjusted_p(col):
    if 'minus' not in col:          # 한 구획 값: 그 구획의 슬라이드·핵 수로 보정
        seg = col.rsplit('_', 1)[1]
        d = pd.DataFrame({'y': P[col], 'trg': P.trg, 'slide': P[f'slide_{seg}'],
                          'lognuc': np.log1p(P[f'nuclei_{seg}'])})
    else:                           # 종양 - 기질 차이: 두 구획 핵 수 합으로 보정
        d = pd.DataFrame({'y': P[col], 'trg': P.trg, 'slide': P.slide_tumor.fillna(P.slide_stroma),
                          'lognuc': np.log1p(P.nuclei_tumor.fillna(0) + P.nuclei_stroma.fillna(0))})
    return smf.ols('y ~ trg + C(slide) + lognuc', data=d.dropna()).fit().pvalues['trg']


rows = []
for col, name in PANELS:
    g = {k: P.loc[P.group == k, col].dropna() for k in ORDER}
    d = P[[col, 'trg']].dropna()
    rho, p = stats.spearmanr(d[col], d.trg)
    rows.append({'지표': name, 'n (Good/Mid/Non)': '/'.join(str(len(g[k])) for k in ORDER),
                 '중앙값 Good': g['Good'].median(), '중앙값 Mid': g['Mid'].median(), '중앙값 Non': g['Non'].median(),
                 'TRG ρ': rho, 'TRG p': p, '보정 p': adjusted_p(col),
                 'p Good-Mid': stats.mannwhitneyu(g['Good'], g['Mid']).pvalue,
                 'p Mid-Non': stats.mannwhitneyu(g['Mid'], g['Non']).pvalue,
                 'p Good-Non': stats.mannwhitneyu(g['Good'], g['Non']).pvalue})
res = pd.DataFrame(rows).set_index('지표')

paired = []
for k in ORDER:
    s = P.loc[P.group == k, ['IgG_tumor', 'IgG_stroma']].dropna()
    paired.append({'반응군': k, 'n': len(s), '기질 - 종양 중앙값': (s.IgG_stroma - s.IgG_tumor).median(),
                   'Wilcoxon p': stats.wilcoxon(s.IgG_tumor, s.IgG_stroma).pvalue})
paired = pd.DataFrame(paired).set_index('반응군')

res.to_csv(f'{OUT}/stats_trend_pairwise.csv')
paired.to_csv(f'{OUT}/stats_IgG_tumor_vs_stroma.csv')
display(res.round(3))
display(paired.round(4))

## 7. 최종 그림

In [ ]:
def fmt(p):
    return 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'


def bracket(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color='black', lw=0.8)
    ax.annotate(text, ((x1 + x2) / 2, y + h), xytext=(0, 1.5), textcoords='offset points',
                ha='center', va='bottom', fontsize=8)


NAMES = {'Good': '좋은 반응', 'Mid': '중간 반응', 'Non': '무반응'}
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flat

for ax, (col, name) in zip(axes, PANELS):
    data = [P.loc[P.group == k, col].dropna() for k in ORDER]
    ax.boxplot(data, showfliers=False)
    for i, d in enumerate(data, 1):
        ax.scatter(rng.normal(i, 0.05, len(d)), d, s=15, alpha=0.7)
    ax.set_xticks([1, 2, 3], [f'{NAMES[k]}\n(n={len(d)})' for k, d in zip(ORDER, data)])
    ax.set_title(f"{name}\nTRG 상관 ρ = {res.loc[name, 'TRG ρ']:.2f}, {fmt(res.loc[name, 'TRG p'])}")
    ax.set_ylabel('log2 차이' if 'minus' in col else 'log2 Q3 정규화 값')
    top = max(d.max() for d in data)
    span = top - min(d.min() for d in data)
    step, h = span * 0.11, span * 0.025
    for j, (a, b, key) in enumerate([(1, 2, 'p Good-Mid'), (2, 3, 'p Mid-Non'), (1, 3, 'p Good-Non')]):
        bracket(ax, a, b, top + step * (0.5 + j), h, fmt(res.loc[name, key]))
    ax.set_ylim(top - span * 1.05, top + step * 3.6)

# 마지막 패널: 같은 환자의 종양 구획 vs 기질 구획 IgG
ax = axes[5]
for gi, k in enumerate(ORDER):
    s = P.loc[P.group == k, ['IgG_tumor', 'IgG_stroma']].dropna()
    x0 = gi * 3
    for _, r in s.iterrows():
        ax.plot([x0, x0 + 1], [r.IgG_tumor, r.IgG_stroma], color='gray', alpha=0.4)
    ax.plot([x0, x0 + 1], [s.IgG_tumor.median(), s.IgG_stroma.median()], color='black', lw=2)
    bracket(ax, x0, x0 + 1, s.max().max() + 0.3, 0.2, fmt(paired.loc[k, 'Wilcoxon p']))
ax.set_xticks([0, 1, 3, 4, 6, 7], ['종양', '기질'] * 3)
ax.set_title('IgG 위치: 같은 환자 종양 vs 기질\n(왼쪽부터 좋은 반응 / 중간 반응 / 무반응)')
ax.set_ylabel('log2 Q3 정규화 값')
ax.set_ylim(top=ax.get_ylim()[1] + 0.8)

fig.text(0.01, 0.005, '막대 위 p: 두 군씩 Mann-Whitney U (다중비교 보정 안 함). '
         '마지막 패널: 같은 환자 종양 vs 기질 Wilcoxon 부호순위 검정.', fontsize=8, color='gray')
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig(f'{OUT}/Fig_GSE279942_IG_by_response.png', dpi=300)
plt.show()

## 8. 재현 기록
실행 환경과 입력·결과를 `final/run_info.json`, `final/requirements_frozen.txt`에 저장

- **패키지 버전**: 주요 패키지 버전 + 전체 `pip freeze`
- **입력 지문**: GEO 파일 3개의 SHA-256과 series matrix의 마지막 수정일
- **결과 지문**: 환자 점수 표의 SHA-256. 다시 실행했을 때 이 값이 같으면 숫자가 완전히 똑같이 재현된 것
- **핵심 수치**: 주요 지표 ρ, p

다른 버전으로 똑같이 맞추려면 `requirements_frozen.txt`에 적힌 버전으로 `pip install 패키지==버전` 후 실행

In [ ]:
import sys, platform, datetime, importlib.metadata as im

with gzip.open(paths['series_matrix'], 'rt', encoding='utf-8', errors='replace') as f:
    geo_update = next((l.split('\t')[1].strip().strip('"') for l in f if l.startswith('!Series_last_update_date')), None)

result_hash = hashlib.sha256(P.round(8).to_csv().encode()).hexdigest()
PKGS = ['numpy', 'pandas', 'scipy', 'statsmodels', 'matplotlib']
run_info = {
    'run_time_KST': datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).isoformat(timespec='seconds'),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'packages': {p: im.version(p) for p in PKGS},
    'seed': SEED,
    'geo_series_last_update': geo_update,
    'input_sha256': file_hash,
    'n_segments': int(meta.shape[0]),
    'n_patients': meta.groupby('group').patient.nunique().to_dict(),
    'patient_scores_sha256': result_hash,
    'key_results': {name: {'rho': round(float(r['TRG ρ']), 4), 'p': round(float(r['TRG p']), 4),
                           'p_adj': round(float(r['보정 p']), 4)} for name, r in res.iterrows()},
}
with open(f'{OUT}/run_info.json', 'w') as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)
frozen = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
with open(f'{OUT}/requirements_frozen.txt', 'w') as f:
    f.write(frozen)

print(json.dumps(run_info, ensure_ascii=False, indent=2))

## 9. 결과 요약 (2026-11-16 실행 기준)
- **IG 신호가 좋은 반응 → 중간 → 무반응 순으로 단계적으로 증가**
  - 기질 IgG: ρ = 0.34, p = 0.023 (보정 p = 0.023) ← 미리 정한 주요 지표
  - 기질 IGKC: ρ = 0.37, p = 0.013 (보정 p = 0.006)
  - 종양 구획 IGKC: ρ = 0.40, p = 0.006 (보정 p = 0.010)
  - 기질 IgA도 같은 방향 (ρ = 0.31, p = 0.042) → IgG만의 변화라기보다 형질세포/IG 전반의 증가
- **IG는 모든 반응군에서 종양 구획보다 기질 구획에 훨씬 많음** (반응군별 Wilcoxon p < 0.001)
- **CD8 종양 − 기질**: 뚜렷한 차이 없음 (ρ = −0.24, p = 0.125). CD8A/CD8B 신호가 대부분 배경 수준이라 이 데이터로는 판단 어려움

## 10. 해석할 때 주의할 점
- **해상도**: 구획 단위라 세포 배정(IG RNA 번짐) 문제의 영향은 없지만, 형질세포 수 증가인지 세포당 발현 증가인지는 구분 불가. 경계로부터의 거리 같은 세포 수준 배치도 볼 수 없음
- **치료**: 원 논문에 환자마다 nCRT 요법이 다를 수 있다고 명시 (단일 요법으로 통일된 코호트가 아님)
- **정상 점막 혼입**: IgA도 같이 증가해서, 무반응군 생검에 정상 점막(IgA 형질세포가 많음)이 더 섞였을 가능성은 아직 확인하지 않음
- **원 논문 결과와의 관계**: 원 논문은 IG를 따로 분석하지 않았음. 기질 GSEA에서 체액성 면역 경로가 반응과 관련된다고만 언급 (방향은 보충 자료 Figure S5 확인 필요)